# Valuaciones todas las empresas (USA)

Un solo notebook para **bancos**, **aseguradoras / financieras no bancarias** y **no financieras**.

El tipo de cada empresa se detecta con su Excel de GuruFocus:
- **aseguradora**: tiene la fila `Total Premiums Earned`
- **banco**: tiene la fila `Total Deposits`
- **no_financiera**: cualquier otro caso

Para correrlo, cambia la lista `TICKETS` y ejecuta todo. Cada empresa va dentro de un `try`:
si una falla, el ciclo sigue con la siguiente y el error queda en la tabla final.

Salidas por empresa (en `CARPETA_SALIDA`): `{ticket}_razones_financieras.xlsx`,
`{ticket}_score_ponderados.xlsx` y `{ticket}_valuaciones_hits_por_anio.xlsx`.

In [1]:
import os
import re
import traceback
from functools import reduce

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

RUTA = "USA/"


def tickets_de_carpeta(ruta=RUTA):
    """Tickets = nombres de los .xlsx de GuruFocus en la carpeta (sin archivos temporales ~$)."""
    return sorted(f[:-len('.xlsx')] for f in os.listdir(ruta)
                  if f.endswith('.xlsx') and not f.startswith('~$'))


TICKETS = tickets_de_carpeta()
CARPETA_SALIDA = "Valuaciones_USA"
ARCHIVO_MERCADO = "Datos históricos del S&P 500 TR"   # csv en la carpeta de trabajo
ARCHIVO_BONOS = "bonos_usa"
ARCHIVO_INFLACION = "inflation_usa"
MARKET_RISK_PREMIUM = 0.07
TASA_CRECIMIENTO_DCF = 0.04   # g2 de los DCF

H_SEMANAS = 156               # horizonte para que el precio toque un objetivo
TOL_OBJETIVO = 0.05           # banda de ±5% alrededor de cada valuación
EXIGIR_HIT_GLOBAL = False     # False: solo hit rate >= 70% en los últimos 10 años (como el código original).
                              # True: además hit rate total >= 70% (como dice el comentario original).
ASEGURADORA_SALUD = False     # True para aseguradoras de salud reguladas (piso MLR)

TIPO_BANCO = "banco"
TIPO_ASEGURADORA = "aseguradora"
TIPO_NO_FIN = "no_financiera"

SECCIONES = [
    'Per Share Data', 'Income Statement', 'Balance Sheet',
    'Cash Flow Statement', 'Valuation Ratios', 'Valuation and Quality',
    'GuruFocus Rankings', 'Ratios'
]
PATRON_SECCIONES = '|'.join(SECCIONES)

MES_MAP = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}

# Tasa promedio anual del bono a 10 años (%). VALORES APROXIMADOS: verificar con FRED (DGS10).
TASA_LIBRE_RIESGO_10Y = {
    2006: 4.79, 2007: 4.63, 2008: 3.66, 2009: 3.26, 2010: 3.22,
    2011: 2.78, 2012: 1.80, 2013: 2.35, 2014: 2.54, 2015: 2.14,
    2016: 1.84, 2017: 2.33, 2018: 2.91, 2019: 2.14, 2020: 0.89,
    2021: 1.45, 2022: 2.95, 2023: 3.96, 2024: 4.21, 2025: 4.30,
}

# Filas que se quitan antes de calificar (no se puntúan o son solo referencia)
RATIOS_A_ELIMINAR = [
    "Net Income", "Sales", "Net Income Growth", "Cost of Goods Sold", "Inventories",
    "Inventories Growth", "Free Cash Flow", "Free cash flow per share", "FCF/DEBT corto plazo",
    "Sales Growth", "Cost of Goods Sold Growth", "Valor en libros",
]

## Comparación de la tasa libre de riesgo

Promedio anual de la serie `DGS1` de `bonos_usa.csv`, para compararlo contra
`TASA_LIBRE_RIESGO_10Y`. La calificación de `Investment Yield(%)` usa este promedio.

In [2]:
def promedio_anual_bonos(ruta="bonos_usa.csv"):
    """Promedio anual de DGS1 (en %) leído de bonos_usa.csv."""
    bonos = pd.read_csv(ruta)
    bonos['observation_date'] = pd.to_datetime(bonos['observation_date'], format='%Y-%m-%d', errors='coerce')
    bonos['DGS1'] = pd.to_numeric(bonos['DGS1'], errors='coerce')
    bonos = bonos.dropna(subset=['observation_date', 'DGS1'])
    return bonos.groupby(bonos['observation_date'].dt.year)['DGS1'].mean().round(4)


promedio_bonos_csv = promedio_anual_bonos()
comparacion_tasas = pd.DataFrame({
    'Año': list(TASA_LIBRE_RIESGO_10Y.keys()),
    'TASA_LIBRE_RIESGO_10Y (%)': list(TASA_LIBRE_RIESGO_10Y.values()),
})
comparacion_tasas['Promedio bonos_usa.csv DGS1 (%)'] = comparacion_tasas['Año'].map(promedio_bonos_csv)
comparacion_tasas['Diferencia (pp)'] = (
    comparacion_tasas['Promedio bonos_usa.csv DGS1 (%)'] - comparacion_tasas['TASA_LIBRE_RIESGO_10Y (%)']
).round(4)
print(comparacion_tasas.to_string(index=False))

 Año  TASA_LIBRE_RIESGO_10Y (%)  Promedio bonos_usa.csv DGS1 (%)  Diferencia (pp)
2006                       4.79                           4.9346           0.1446
2007                       4.63                           4.5302          -0.0998
2008                       3.66                           1.8463          -1.8137
2009                       3.26                           0.4715          -2.7885
2010                       3.22                           0.3192          -2.9008
2011                       2.78                           0.1810          -2.5990
2012                       1.80                           0.1746          -1.6254
2013                       2.35                           0.1321          -2.2179
2014                       2.54                           0.1202          -2.4198
2015                       2.14                           0.3179          -1.8221
2016                       1.84                           0.6166          -1.2234
2017            

## Pesos de importancia por tipo de empresa

In [3]:
IMPORTANCIA = {
    TIPO_BANCO: {
        'Net Profit Margin(%)': 4, 'EPS': 1, 'Revenue Per Share': 1, '# Shares': 1,
        'Net Interest Margin (Bank Only) %': 4, 'Dividends per Share': 1,
        'Free cash flow to sales(%)': 4, 'Free Cash Flow to Debt ratio': 3,
        'Dividend Payout Ratio(%)': 2, 'net worth to long-term debt ratio': 2,
        'Pasivo/Fondos Propios': 5, 'long-term debt-to-equity(%)': 4,
        'Gasto financiero(%)': 3, 'PEGY Ratio': 1, 'PER': 3, 'P/S Ratio': 2,
        'ROA%': 6, 'ROE%': 8, 'ROB': 2, 'WACC %': 2, 'WACC % Growth': 1,
        'Piotrivski F-Score': 7, 'Beneish M-Score': 5, 'Return-on-Tangible-Equity': 6,
        'Efficiency Ratio(%)': 7, 'Equity-to-Asset': 7, 'Loan to Deposit Ratio(%)': 8,
    },
    TIPO_ASEGURADORA: {
        'Net Profit Margin(%)': 5, 'EPS': 4, 'Revenue Per Share': 2, '# Shares': 0.5,
        'Dividends per Share': 1, 'Free cash flow to sales(%)': 8,
        'Free Cash Flow to Debt ratio': 5, 'Dividend Payout Ratio(%)': 1,
        'net worth to long-term debt ratio': 3, 'Pasivo/Fondos Propios': 3,
        'long-term debt-to-equity(%)': 5, 'PEGY Ratio': 1, 'PER': 5, 'P/S Ratio': 2,
        'ROA%': 4, 'ROE%': 6, 'ROB': 1.5, 'ROIC %': 10, 'WACC %': 0,
        'Piotrivski F-Score': 5, 'Altman Z score': 0, 'Beneish M-Score': 1,
        'Loss Ratio(%)': 6, 'Expense Ratio(%)': 4, 'Investment Yield(%)': 5,
        'Reserve-to-Premium Ratio': 3, 'Book Value Growth(%)': 6, 'Float': 0,
        'Float Growth(%)': 3,
    },
    TIPO_NO_FIN: {
        'Amortizacion(%)': 1, 'Net Profit Margin(%)': 5, 'Gross Profit(%)': 4,
        'Operating Margin(%)': 3, 'EPS': 4, 'Revenue Per Share': 2, '# Shares': 0.5,
        'Inventory Turnover Ratio': 2, 'Dividends per Share': 1,
        'Free cash flow to sales(%)': 8, 'Free Cash Flow to Debt ratio': 5,
        'Dividend Payout Ratio(%)': 1, 'net worth to long-term debt ratio': 3,
        'Pasivo/Fondos Propios': 3, 'Current Ratio': 1, 'long-term debt-to-equity(%)': 5,
        'Gasto financiero(%)': 3, 'Inventory to current Assets(%)': 1,
        'Net Trading Cycle': 3, 'PEGY Ratio': 3, 'PER': 5, 'P/S Ratio': 2,
        'Receivable to Current Assets(%)': 1, 'ROA%': 4, 'ROE%': 6, 'ROB': 1.5,
        'ROIC %': 10, 'WACC %': 0, 'Test de Acidez': 1, 'Piotrivski F-Score': 5,
        'Altman Z score': 5, 'Beneish M-Score': 1,
    },
}

## Carga del Excel y detección del tipo de empresa

In [4]:
def cargar_excel(ticket):
    """Lee el Excel de GuruFocus y quita las filas de encabezado de sección."""
    df_err = pd.read_excel(f"{RUTA}{ticket}.xlsx", header=19)
    primera_col = df_err.columns[0]
    mask = df_err[primera_col].astype(str).str.contains(PATRON_SECCIONES, case=False, na=False)
    excepcion = df_err[primera_col].astype(str).str.strip() == 'Balance Sheet Cash And Cash Equivalents'
    df_err = df_err[~(mask & ~excepcion)].reset_index(drop=True)

    df_err.columns = df_err.iloc[0]
    if '\t' in df_err.columns:
        df_err = df_err.drop(columns=['\t'], errors='ignore')
    return df_err[1:].reset_index(drop=True)


def detectar_tipo(df_err):
    """Clasifica la empresa según las filas que trae su Excel."""
    filas = set(df_err.iloc[:, 0].astype(str).str.strip())
    if 'Total Premiums Earned' in filas:
        return TIPO_ASEGURADORA
    if 'Total Deposits' in filas:
        return TIPO_BANCO
    return TIPO_NO_FIN


def limpiar_df_financiero(df):
    """'-' pasa a NaN y todas las columnas (menos Fiscal Period) quedan numéricas."""
    # df = df.copy().replace('-', np.nan)
    # cols_datos = df.columns.drop('Fiscal Period')
    # df[cols_datos] = df[cols_datos].apply(pd.to_numeric, errors='coerce')
    # return df
    df = df.copy()

    # Eliminar columnas con nombre duplicado (conserva la primera)
    df = df.loc[:, ~df.columns.duplicated()]

        # Reemplazar '-' por NaN en todo el DF
    df.replace('-', np.nan, inplace=True)

        # Todas las columnas excepto 'Fiscal Period' a numéricas
    cols_datos = df.columns.drop('Fiscal Period')

    df[cols_datos] = df[cols_datos].apply(
            pd.to_numeric, errors='coerce'
        )

    return df


def preparar_periodos(df_err):
    """Define los periodos (últimos 20 años + TTM) y arma los df anual y trimestral."""
    cols = df_err.columns.tolist()
    ttm_idx = cols.index("TTM/current")
    cols_trim = [c for c in cols[ttm_idx + 1:] if re.search(r'[A-Za-z]+\d{4}', str(c))]
    m = re.search(r'([A-Za-z]+)(\d{4})', cols_trim[-1])
    nombre_ttm = f"{m.group(1)}{m.group(2)}"

    df_err = df_err.rename(columns={"TTM/current": nombre_ttm})
    ttm_idx = df_err.columns.tolist().index(nombre_ttm)

    df_hist = df_err.iloc[:, :ttm_idx].copy()
    row = df_hist.loc[df_hist["Fiscal Period"].eq("Revenue per Share")].iloc[0]
    valid_headers = [
        col for col in df_hist.columns
        if col != "Fiscal Period" and not pd.isna(row[col])
        and not (isinstance(row[col], str) and row[col].strip() == "-")
    ] + [nombre_ttm]

    valid_years = [int(h[-4:]) for h in valid_headers]
    last_20_years = valid_years[-20:]
    last_20_years_mes = valid_headers[-20:]

    # Anual: últimos 20 años + TTM (TTM al final)
    df_anual = df_err.iloc[:, :ttm_idx + 1]
    anio_limite = min(last_20_years)
    cols_anio = []
    for c in df_anual.columns:
        if c in ('Fiscal Period', nombre_ttm):
            continue
        m_anio = re.search(r'\d{4}', str(c))
        if m_anio and int(m_anio.group()) >= anio_limite:
            cols_anio.append(c)
    df_anual_filtrado = df_anual[['Fiscal Period'] + cols_anio + [nombre_ttm]].fillna(0)
    df_anual_filtrado = limpiar_df_financiero(df_anual_filtrado).dropna(axis=1, how='all')

    # Trimestral: solo dentro del rango de años
    df_trimestral = df_err.iloc[:, ttm_idx + 1:].copy()
    df_trimestral.insert(0, 'Fiscal Period', df_err['Fiscal Period'])
    anio_min, anio_max = min(last_20_years), max(last_20_years)
    cols_trim_sel = []
    for c in df_trimestral.columns:
        if c == 'Fiscal Period':
            continue
        m_anio = re.search(r'\d{4}', str(c))
        if m_anio and anio_min <= int(m_anio.group()) <= anio_max:
            cols_trim_sel.append(c)
    cols_trim_sel.sort(key=lambda c: int(re.search(r'\d{4}', str(c)).group()))
    df_trimestral_filtrado = df_trimestral[['Fiscal Period'] + cols_trim_sel].fillna(0)
    df_trimestral_filtrado = limpiar_df_financiero(df_trimestral_filtrado).dropna(axis=1, how='all')

    return {
        'nombre_ttm': nombre_ttm,
        'valid_headers': valid_headers,
        'last_20_years': last_20_years,
        'last_20_years_mes': last_20_years_mes,
        'df_anual_filtrado': df_anual_filtrado,
        'df_trimestral_filtrado': df_trimestral_filtrado,
    }

## Utilidades de fechas y series

In [5]:
def ordenar_por_fecha(serie):
    """Ordena una serie cuyo índice son etiquetas 'MmmYYYY' por fecha real."""
    fechas = pd.to_datetime(pd.Index(serie.index).astype(str), format='%b%Y', errors='coerce')
    return serie.iloc[np.argsort(fechas.values)]


def fecha_fin_de_mes(etiquetas):
    return pd.to_datetime(etiquetas, format='%b%Y') + pd.offsets.MonthEnd(0)


def anio_de(etiqueta):
    return int(re.search(r'\d{4}', str(etiqueta)).group())


def serie_fila(df, nombre):
    """Valores de una fila de df_anual_filtrado o df_trimestral_filtrado."""
    return df.loc[df['Fiscal Period'] == nombre].iloc[0, 1:]


def serie_a_df_anual(serie, nombre_columna):
    """Serie con índice 'MmmYYYY' -> DataFrame (Año, valor) ordenado por fecha."""
    s = ordenar_por_fecha(pd.to_numeric(serie, errors='coerce'))
    df = pd.DataFrame({'Año': s.index.astype(str), nombre_columna: s.values})
    return df.dropna(subset=[nombre_columna]).reset_index(drop=True)


def leer_precios(ruta, formato_fecha):
    """Lee un csv de precios diarios (Fecha, Cierre, ...) con Cierre numérico."""
    df = pd.read_csv(ruta)
    df['Fecha'] = pd.to_datetime(df['Fecha'], format=formato_fecha, errors='coerce')
    df['Cierre'] = pd.to_numeric(
        df['Cierre'].astype(str).str.replace(',', '', regex=False).str.strip(), errors='coerce'
    )
    return df.dropna(subset=['Fecha', 'Cierre']).sort_values('Fecha').reset_index(drop=True)

## Datos de mercado: S&P 500, bonos, inflación, beta y precios de la empresa

In [6]:
def cargar_mercado(ticket, per):
    headers, last_20 = per['valid_headers'], per['last_20_years']

    # Bono a 1 año (promedio de los últimos 12 meses), en decimal
    bonos = pd.read_csv(f"{ARCHIVO_BONOS}.csv")
    bonos['observation_date'] = pd.to_datetime(bonos['observation_date'], format='%Y-%m-%d')
    bonos = bonos.rename(columns={bonos.columns[1]: 'DGS1'})
    bonos['DGS1'] = pd.to_numeric(bonos['DGS1'], errors='coerce')
    bonos = bonos[bonos['observation_date'].dt.year.isin(last_20)]
    filas = []
    for mes_anio in headers:
        ref = fecha_fin_de_mes(mes_anio)
        sub = bonos.loc[(bonos['observation_date'] > ref - relativedelta(years=1))
                        & (bonos['observation_date'] <= ref), 'DGS1']
        if not sub.empty:
            filas.append({'Año': mes_anio, 'Tasa_Libre_Riesgo': round(sub.mean() / 100, 6)})
    tasa = pd.DataFrame(filas)

    # Inflación anual (promedio mensual de CPIAUCSL_PC1) y factor a 3 años
    inf = pd.read_csv(f"{ARCHIVO_INFLACION}.csv")
    inf['observation_date'] = pd.to_datetime(inf['observation_date'], dayfirst=True, errors='coerce')
    inf['CPIAUCSL_PC1'] = pd.to_numeric(inf['CPIAUCSL_PC1'], errors='coerce')
    inf = inf[inf['observation_date'].dt.year.isin(last_20)]
    df_inf = (
        inf.assign(Año=inf['observation_date'].dt.year)
        .groupby('Año', as_index=False)['CPIAUCSL_PC1'].mean()
        .rename(columns={'CPIAUCSL_PC1': 'Tasa de Inflación'})
        .sort_values('Año').reset_index(drop=True)
    )
    df_inf['Tasa de Inflación'] = (df_inf['Tasa de Inflación'] / 100).round(4)
    df_inf['Factor_Inflacion'] = ((1 + df_inf['Tasa de Inflación']) ** 3).round(4)

    # Precios de la empresa y del S&P 500
    precios = leer_precios(f"{RUTA}DatosHistoricos/Datos históricos {ticket}.csv", '%Y-%m-%d')
    mercado = leer_precios(f"{ARCHIVO_MERCADO}.csv", '%d.%m.%Y')

    # Beta: regresión de rendimientos diarios de la empresa contra el S&P, ventana de 1 año
    rend_stock = precios.assign(Rend_Stock=precios['Cierre'].pct_change()).dropna(subset=['Rend_Stock'])
    rend_mkt = mercado.assign(Rend_Market=mercado['Cierre'].pct_change()).dropna(subset=['Rend_Market'])
    df_beta = pd.merge_asof(
        rend_stock[['Fecha', 'Rend_Stock']].sort_values('Fecha'),
        rend_mkt[['Fecha', 'Rend_Market']].sort_values('Fecha'),
        on='Fecha', direction='nearest'
    )
    filas = []
    for mes_anio in headers:
        ref = fecha_fin_de_mes(mes_anio)
        g = df_beta[(df_beta['Fecha'] > ref - relativedelta(years=1)) & (df_beta['Fecha'] <= ref)]
        if len(g) < 10:
            continue
        var_m = g['Rend_Market'].var()
        if pd.isna(var_m) or var_m == 0:
            continue
        filas.append({'Año': mes_anio, 'Beta': g['Rend_Stock'].cov(g['Rend_Market']) / var_m})

    return {
        'tasa': tasa,
        'inflacion': df_inf,
        'beta': pd.DataFrame(filas),
        'precios': precios,
    }

## Razones financieras

Las funciones `agregar_*` reciben el DataFrame de razones (`rf`) como primer argumento.
Cada tipo de empresa tiene su núcleo común y sus razones propias.

In [7]:
def _n_anios(rf):
    return len(rf.columns) - 2


def _na(rf, nombre, calculo="", explicacion=""):
    rf.loc[nombre] = [calculo, explicacion] + ['N/A'] * _n_anios(rf)


def _falta(df, fila):
    return not (df['Fiscal Period'] == fila).any()


def agregar_razon(rf, df, numerador, nombre, *denominadores,
                  como_porcentaje=True, calculo="", explicacion=""):
    """Razón = suma(numerador) / suma(denominadores). Acepta un numerador str o lista."""
    numeradores = [numerador] if isinstance(numerador, str) else list(numerador)
    denominadores = list(denominadores)
    requeridas = numeradores + denominadores
    if any(_falta(df, f) for f in requeridas):
        _na(rf, nombre, calculo, explicacion)
        return

    # Un numerador str toma la primera fila con ese nombre (como en los notebooks originales);
    # una lista suma sus filas.
    if isinstance(numerador, str):
        num = df.loc[df['Fiscal Period'] == numerador].iloc[0, 1:].astype(float)
    else:
        num = df.loc[df['Fiscal Period'].isin(numeradores)].iloc[:, 1:].astype(float).sum(axis=0)
    if denominadores:
        den = df.loc[df['Fiscal Period'].isin(denominadores)].iloc[:, 1:].astype(float).sum(axis=0)
    else:
        den = pd.Series(1.0, index=num.index)

    # Si una fila de deuda suma 0, la razón se marca como "No deuda"
    filas_deuda = [f for f in requeridas if any(p in f.lower() for p in ('debt', 'deuda'))]
    es_cero_deuda = pd.Series(False, index=num.index)
    if filas_deuda:
        deuda = df.loc[df['Fiscal Period'].isin(filas_deuda)].iloc[:, 1:].astype(float).sum(axis=0)
        es_cero_deuda = deuda == 0

    mult = 100 if como_porcentaje else 1
    valores = []
    for i, (n, d) in enumerate(zip(num.values, den.values)):
        v = n / d * mult if d != 0 else np.nan
        if es_cero_deuda.iloc[i]:
            valores.append("No deuda")
        elif pd.notna(v):
            valores.append(f"{v:.2f}")
        else:
            valores.append("N/A")
    rf.loc[nombre] = [calculo, explicacion] + valores


def agregar_razon_resta(rf, df, nombre, denominador, num1, num2,
                        como_porcentaje=True, calculo="", explicacion=""):
    """Razón = (num1 - num2) / denominador. Usada para el test de acidez."""
    if any(_falta(df, f) for f in (denominador, num1, num2)):
        _na(rf, nombre, calculo, explicacion)
        return
    n1 = df.loc[df['Fiscal Period'] == num1].iloc[0, 1:].astype(float).values
    n2 = df.loc[df['Fiscal Period'] == num2].iloc[0, 1:].astype(float).values
    d = df.loc[df['Fiscal Period'] == denominador].iloc[0, 1:].astype(float).values
    mult = 100 if como_porcentaje else 1
    valores = [f"{(a - b) / c * mult:.2f}" if c != 0 else "no deuda" for a, b, c in zip(n1, n2, d)]
    rf.loc[nombre] = [calculo, explicacion] + valores


def agregar_fila(rf, df, fila, nombre, calculo="", explicacion=""):
    """Copia una fila tal cual de df_anual_filtrado."""
    if _falta(df, fila):
        _na(rf, nombre, calculo, explicacion)
        return
    valores = list(df.loc[df['Fiscal Period'] == fila].iloc[0, 1:].values)
    rf.loc[nombre] = [calculo, explicacion] + valores


def agregar_crecimiento(rf, base, nombre, calculo="", explicacion=""):
    """Crecimiento % vs año anterior de una fila ya calculada en rf."""
    if base not in rf.index:
        _na(rf, nombre, calculo, explicacion)
        return
    serie = pd.to_numeric(rf.loc[base].iloc[2:], errors='coerce')
    crec = (serie.pct_change() * 100).fillna(0)
    rf.loc[nombre] = [calculo, explicacion] + [f"{x:.2f}" for x in crec]


def agregar_dividir(rf, num, den, nombre, como_porcentaje=True, calculo="", explicacion=""):
    """Divide dos filas ya calculadas en rf."""
    if num not in rf.index or den not in rf.index:
        _na(rf, nombre, calculo, explicacion)
        return
    n = pd.to_numeric(rf.loc[num].iloc[2:], errors='coerce').values
    d = pd.to_numeric(rf.loc[den].iloc[2:], errors='coerce').values
    r = n / d
    if como_porcentaje:
        r = r * 100
    r = pd.Series(r).fillna(0).values
    rf.loc[nombre] = [calculo, explicacion] + [f"{x:.2f}" for x in r]


def calcular_razones(df, tipo):
    cols = [c for c in df.columns if c != 'Fiscal Period']
    rf = pd.DataFrame(columns=['Calculo', 'Explicacion'] + cols)

    # ---------- Núcleo común a los tres tipos ----------
    agregar_razon(rf, df, 'Net Income', 'Net Profit Margin(%)', 'Revenue', calculo="Net income / sales",
                  explicacion="* Algo mayor a 20% indica una ventaja \n* Algo menor a 10% puede indicar mucha competencia\n* empresas que generen entre un 10 al 20% son buenas\n* los bancos suelen tener porcentajes más altos")
    agregar_fila(rf, df, 'Net Income', 'Net Income', explicacion="Se busca tendencia alcista")
    agregar_razon(rf, df, 'Net Income', 'EPS', 'Shares Outstanding (Diluted Average)', como_porcentaje=False,
                  calculo="Net Income/Shares", explicacion="Se busca tendencia a la alza en los ultimos 5 0 10 años")
    agregar_razon(rf, df, 'Revenue', 'Revenue Per Share', 'Shares Outstanding (Diluted Average)',
                  como_porcentaje=False, calculo="Revenue/Shares", explicacion="Se busca tendencia alcista")
    agregar_fila(rf, df, 'Shares Outstanding (Diluted Average)', '# Shares', explicacion="Se busca tendencia bajista o que no suba")
    agregar_fila(rf, df, 'Revenue', 'Sales', explicacion="Se busca tendencia alcista")
    agregar_crecimiento(rf, 'Net Income', 'Net Income Growth', calculo="Crecimiento vs año pasado",
                        explicacion="Se busca porcentajes estables o en crecimiento")
    agregar_crecimiento(rf, 'Sales', 'Sales Growth', calculo="Crecimiento vs año pasado",
                        explicacion="Se busca porcentajes estables o en crecimiento")
    agregar_fila(rf, df, 'Dividends per Share', 'Dividends per Share', explicacion="Buscar tendencia alcista")
    agregar_fila(rf, df, 'Free Cash Flow', 'Free Cash Flow', explicacion="Buscar una tendencia alcista")
    agregar_dividir(rf, 'Free Cash Flow', 'Sales', 'Free cash flow to sales(%)', True, calculo="FCF/Sales",
                    explicacion="Si el porcentaje da algo en torno al 15% puede indicar un foso económico")
    agregar_dividir(rf, 'Free Cash Flow', '# Shares', 'Free cash flow per share', False, calculo="FCF/Shares",
                    explicacion="Se busca tendencia alcista")
    agregar_razon(rf, df, 'Free Cash Flow', 'Free Cash Flow to Debt ratio',
                  'Short-Term Debt & Capital Lease Obligation', 'Long-Term Debt & Capital Lease Obligation',
                  como_porcentaje=False, calculo="FCF/Total Debt", explicacion="Entre más cerca de 1 mejor")
    agregar_razon(rf, df, 'Free Cash Flow', 'FCF/DEBT corto plazo', 'Short-Term Debt & Capital Lease Obligation',
                  como_porcentaje=False, calculo="FCF/DEBT corto plazo", explicacion="Entre más cerca o alejado de 1 mejor")
    agregar_dividir(rf, 'Dividends per Share', 'EPS', 'Dividend Payout Ratio(%)', True,
                    calculo="Dividend earnings per share/ earnings per share",
                    explicacion="Buscar empresas que tenga un ratio de entre 20 y 60%")
    agregar_razon(rf, df, 'Net Income', 'net worth to long-term debt ratio', 'Long-Term Debt & Capital Lease Obligation',
                  como_porcentaje=False, calculo="Net income/ Long Term Debt",
                  explicacion="Igual a 1 esta bien, pero mayor a 3 indica ventaja competitiva")
    agregar_razon(rf, df, 'Total Liabilities', 'Pasivo/Fondos Propios', 'Total Equity', como_porcentaje=False,
                  calculo="total liabilities/total equity",
                  explicacion="Si algo es menor a 1 es mejor\n* Para las financiera un relación menor a .80 es una ventaja competitiva")
    agregar_razon(rf, df, 'Long-Term Debt & Capital Lease Obligation', 'long-term debt-to-equity(%)', 'Total Equity',
                  como_porcentaje=True, calculo="Long term debt/equity",
                  explicacion="Se busca un balance de 75% capital, 25% deuda\nSe busca resultados menores a 25%")
    agregar_fila(rf, df, 'PEG Ratio', 'PEGY Ratio', calculo="P/E ratio/ EPS growth rate+dividend yield",
                 explicacion="*No es efectivo para empresas que pagan más de un 8% de dividendo,Buscar que sea menor a 1")
    agregar_fila(rf, df, 'PE Ratio', 'PER', calculo="Price* shares outstandings / net income",
                 explicacion="* No invertir en acciones con PER promedio mayor a 25\n* PER menor a 9 es bueno y más una acción dentro de un indice\n*una acción se puede vender cuando ofrecer PER mayores a 40, sobre todo en empresas grandes\n*Cuando una empresa crece al mismo risto pero su PER promedio baja puede ser interesante")
    agregar_fila(rf, df, 'PS Ratio', 'P/S Ratio', calculo="market cap / sales",
                 explicacion="*Se usa en empresas con perdidas temporales\n* solo usarlo para comparar entre la misma empresa\n*si el ratio es bajo con respecto a su promedio historico podria ser buena opción de compra\n*Entre mayor sea indica que los inversionistas estan dispuestos a pagar más por cada venta y por ende aumentara el precio de la accion")
    agregar_fila(rf, df, 'ROA %', 'ROA%', calculo="net income 2020/((total assets 2019+total assets 2020)/2)",
                 explicacion="* Una empresa no financiera que genere un ROA del 7% es buena\n* Un ROA mayor al 20% es Excelente")
    agregar_fila(rf, df, 'ROE %', 'ROE%', calculo="net income 2020/((shareholders equity 2019+shareholders equity 2020)/2)",
                 explicacion="*Un ROE mayor a 15% es bueno si es mayor al 20% puede indicar un gran foso\n* Las financieras se debe buscar algo mayor al 12%")
    agregar_razon(rf, df, 'Total Assets', 'ROB', 'Total Equity', como_porcentaje=False,
                  calculo="Total assets 2020/ ((shareholders equity 2019+shareholders equity 2020)/2)",
                  explicacion="ratios mayores a 4.5 vuelve a las empresas riesgosas")
    if tipo != TIPO_BANCO:
        agregar_fila(rf, df, 'ROIC %', 'ROIC %',
                     calculo="Operating income * (1-tax rate %) / ((total assets - accounts payable y acrued expense - (cash, cash equivalents y marketable securities - max(0,total current liabilities - total current assets + cash,cash equivalents, marketable securities))",
                     explicacion="Si es mayor a su WACC indica que genera más dinero que lo que paga por el\nSi tiene un ROIC mayor al 15% tiene una ventaja competitiva")
    agregar_fila(rf, df, 'WACC %', 'WACC %', calculo="",
                 explicacion="Costo que tiene la obtencion del capital, entre menor sea mejor"
                             + ("" if tipo == TIPO_BANCO else ",debe ser menor al ROIC"))
    agregar_fila(rf, df, 'Book Value per Share', 'Valor en libros',
                 calculo="Stockholders equity - preferred stock / average shares outstanding",
                 explicacion="*Comprar acciones que no pase su valor en 1.5\n* Mas util en financiera debido a lo activo de sus liquidos\n*Una empresa con un Valor en libros bajo, pero un ROE alto puede ser una ganga")
    agregar_fila(rf, df, 'Piotroski F-Score', 'Piotrivski F-Score',
                 explicacion="* Un buen score es en 7,8,9 sobre su estabilidad finaciera\n* las zonas grises esta entre 4,5,6\n* Mal o bajo score entre 0,1,2,3")
    if tipo != TIPO_BANCO:
        agregar_fila(rf, df, 'Altman Z-Score', 'Altman Z score',
                     explicacion="*Algo menor a 1.8 es malo ya que indica que la empresa podria entrar en bancarrota en los proximos dos años\n* entre 1.8 y 3 es zona griss\n* mayor a 3 es zona segura e indica una alta fuerza financiera")
    agregar_fila(rf, df, 'Beneish M-Score', 'Beneish M-Score',
                 explicacion="* Menor a 1.78 indica que podrian manipular sus estados financieros.\n* Mayor a 1.78, podria indicar que no manipulan sus estados financieros")

    # ---------- Razones propias de cada tipo ----------
    if tipo == TIPO_BANCO:
        agregar_fila(rf, df, 'Net Interest Margin (Bank Only) %', 'Net Interest Margin (Bank Only) %',
                     explicacion="Mayor a 3.5 % es bueno , una tendencia a la baja es mala")
        agregar_razon(rf, df, 'Net Interest Income (for Banks)', 'Gasto financiero(%)', 'Pre-Tax Income',
                      calculo="net interest income(expenses)/operating income",
                      explicacion="* Un gasto menor a 15% indica una ventaja\n* En el area financiera un porcentaje bajo puede ser un 30%\n*En cualquier industria la empresa con la proporción más baja tiene la ventaja")
        agregar_crecimiento(rf, 'WACC %', 'WACC % Growth', calculo="Crecimiento vs año pasado",
                            explicacion="Se busca porcentajes estables o en bajada")
        agregar_fila(rf, df, 'Return-on-Tangible-Equity', 'Return-on-Tangible-Equity',
                     explicacion="* Mayor a 15% es bueno,10% es aceptable,< 5% es malo")
        agregar_razon(rf, df, 'Total Noninterest Expense', 'Efficiency Ratio(%)',
                      'Net Interest Income (for Banks)', 'Non Interest Income', como_porcentaje=True,
                      calculo="Total Noninterest Expense / (Net Interest Income + Non Interest Income)",
                      explicacion="Menor a 50% es excelente, menor a 60% es bueno,mayor a 65-70% indica una estructura de costos alta")
        agregar_fila(rf, df, 'Equity-to-Asset', 'Equity-to-Asset',
                     explicacion="* Mayor a 10% es bueno, menor a 6% indica apalancamiento")
        agregar_razon(rf, df, 'Net Loan', 'Loan to Deposit Ratio(%)', 'Total Deposits', como_porcentaje=True,
                      calculo="Net Loan / Total Deposits",
                      explicacion="Entre 80-90 % es bueno, mayor a 100% indica que la empresa esta prestando más de lo que tiene en depósitos y menor a 70% indica que la empresa no esta prestando lo suficiente")

    elif tipo == TIPO_ASEGURADORA:
        agregar_razon(rf, df, 'Net Policyholder Benefits/Claims', 'Loss Ratio(%)', 'Total Premiums Earned',
                      como_porcentaje=True,
                      calculo="Net Policyholder Benefits/Claims / Total Premiums Earned",
                      explicacion="Menor a 60% → excelente (1.0)60-70% → bueno (0.75)70-80% → aceptable, zona gris (0.5)80-90% → ajustado (0.25)Mayor a 90% → sin margen (0.0)")
        agregar_razon(rf, df, ['Selling, General, & Admin. Expense', 'Policy Acquisition Expense'], 'Expense Ratio(%)',
                      'Total Premiums Earned', como_porcentaje=True,
                      calculo="(Selling, General & Admin. Expense + Policy Acquisition Expense) / Total Premiums Earned",
                      explicacion="Menor a 12% → excelente (1.0)12-15% → bueno (0.75)15-20% → aceptable (0.5)20-25% → pesado (0.25)Mayor a 25% → ineficiente (0.0)")
        agregar_razon(rf, df, 'Net Investment Income', 'Investment Yield(%)',
                      'Fixed Maturity Investment', 'Equity Investments', 'Short-term investments',
                      como_porcentaje=True,
                      calculo="Net Investment Income / (Fixed Maturity Investment + Equity Investments + Short-term investments)",
                      explicacion="+1.5pts → excelente (1.0)≥ +0.5pts → bueno (0.75)±0.5pts de la tasa → neutro (0.5)hasta -1.5pts → débil (0.25)más de -1.5pts por debajo → malo (0.0)")
        agregar_razon(rf, df, 'Unpaid Loss & Loss Reserve', 'Reserve-to-Premium Ratio', 'Total Premiums Earned',
                      como_porcentaje=True,
                      calculo="Unpaid Loss & Loss Reserve / Total Premiums Earned",
                      explicacion="Estable o creciendo (-5% a +15%) → sano (1.0)\nCaída moderada (-5% a -15%) sin razón de negocio → revisar (0.5)\nCaída fuerte (mayor a -15%) → posible sub-reserva para inflar utilidad (0.0)\nAumento muy fuerte (mayor a +30%) → también amerita revisión, no es automáticamente malo (0.5)")
        agregar_crecimiento(rf, 'Valor en libros', 'Book Value Growth(%)', calculo="Crecimiento vs año pasado",
                            explicacion="Mayor a 12% → crecimiento tipo Berkshire (1.0)\n8-12% → bueno (0.75)\n4-8% → aceptable (0.5)\n0-4% → débil (0.25)\nNegativo → destrucción de valor (0.0)")
        # Float = policyholder funds + primas no devengadas + beneficios futuros + reservas de siniestros
        campos_float = ['Policyholder Funds', 'Unearned Premiums', 'Future Policy Benefits', 'Unpaid Loss & Loss Reserve']
        formula_float = "Policyholder Funds + Unearned Premiums + Future Policy Benefits + Unpaid Loss & Loss Reserve"
        if not any(_falta(df, c) for c in campos_float):
            total = sum(df.loc[df['Fiscal Period'] == c].iloc[0, 1:].astype(float) for c in campos_float)
            rf.loc['Float'] = [formula_float, "Float total de la aseguradora (capital de terceros disponible para invertir)"] + list(total.values)
        else:
            rf.loc['Float'] = [formula_float, "Float total de la aseguradora"] + ['N/A'] * _n_anios(rf)
        agregar_crecimiento(rf, 'Float', 'Float Growth(%)', calculo="Crecimiento vs año pasado",
                            explicacion="Mayor a 8% → excelente (1.0)\n3-8% → bueno (0.75)\n0-3% → estancado (0.5)\n-5% a 0% → leve contracción (0.25)\nMenor a -5% → el float se evapora (0.0)")

    else:  # no financiera
        agregar_razon(rf, df, 'Depreciation, Depletion and Amortization', 'Amortizacion(%)', 'Gross Profit',
                      como_porcentaje=True, calculo="Amortization/ gross profit",
                      explicacion="* Una empresa con menor a 10% es bueno\n* Más del 20% indica mucha competencia en el sector")
        agregar_razon(rf, df, 'Gross Profit', 'Gross Profit(%)', 'Revenue', como_porcentaje=True,
                      calculo="Gross profit/sales",
                      explicacion="* Un porcentaje mayor a 40% indica cierta ventaja\n* Porcentajes menores a 40% en un sector significa demasiada competencia y algo menor al 20% en una industria indica dificultad de crear ventaja")
        agregar_razon(rf, df, 'Operating Income', 'Operating Margin(%)', 'Revenue', como_porcentaje=True,
                      calculo="Operating profit/sales", explicacion="Se busca tendencia alcista o estable")
        agregar_fila(rf, df, 'Cost of Goods Sold', 'Cost of Goods Sold', calculo="",
                     explicacion="Se busca tendencia bajista o que incrementen menos que las sales")
        agregar_crecimiento(rf, 'Cost of Goods Sold', 'Cost of Goods Sold Growth', calculo="Crecimiento vs año pasado",
                            explicacion="Se busca que crezca a menor porcentaje que las Sales")
        agregar_fila(rf, df, 'Total Inventories', 'Inventories', calculo="", explicacion="se busca crecimiento estable")
        agregar_crecimiento(rf, 'Inventories', 'Inventories Growth', calculo="Crecimiento vs año pasado",
                            explicacion="Se busca crecimiento estable")
        agregar_dividir(rf, 'Cost of Goods Sold', 'Inventories', 'Inventory Turnover Ratio', False,
                        calculo="Cost of Goods Sold / Inventories",
                        explicacion="*Un valor mayor a 8 indica excelencia\n *Un valor entre 4 y 8 indica rango saludable\n*Menor a 4 indica baja rotación")
        agregar_razon(rf, df, '  Interest Expense', 'Gasto financiero(%)', 'Operating Income', como_porcentaje=True,
                      calculo="net interest income(expenses)/operating income",
                      explicacion="* Un gasto menor a 15% indica una ventaja\n* En el area financiera un porcentaje bajo puede ser un 30%\n*En cualquier industria la empresa con la proporción más baja tiene la ventaja")
        agregar_razon(rf, df, 'Total Current Assets', 'Current Ratio', 'Total Current Liabilities', como_porcentaje=False,
                      explicacion="*En teoría algo mayor a 1 indica solvencia, pero existen empresas con algo menor a 1 que debido a su rentabilidad pueden cubrir el pasivo\n*Algo mayor a 1.5 puede ser muy bueno\n*evitar aquellas menores a .5")
        agregar_razon(rf, df, 'Total Inventories', 'Inventory to current Assets(%)', 'Total Current Assets',
                      como_porcentaje=True, calculo="inventories/current asset",
                      explicacion="Buscar compañias con un rango menor al 40% y evitar empresas con algo mayor al 60%")
        agregar_fila(rf, df, 'Cash Conversion Cycle', 'Net Trading Cycle',
                     calculo="Rotación de cuentas por cobrar + Rotación de inventarios - Rotación de cuentas por pagar",
                     explicacion="Entre menor sea, mejor, buscar una tendencia bajista,un valor negativo es bueno siempre y cuando el revenue continue subiendo")
        agregar_razon(rf, df, '  Accounts Receivable', 'Receivable to Current Assets(%)', 'Total Current Assets',
                      como_porcentaje=True, calculo="accounts receivable/current asset",
                      explicacion="Buscar compañias con un rango menor al 40% y evitar empresas con algo mayor al 60%")
        agregar_razon_resta(rf, df, 'Test de Acidez', 'Total Current Liabilities', 'Total Current Assets', 'Total Inventories',
                            como_porcentaje=False, calculo="(Current assets-inventory)/current liabilities",
                            explicacion="Un indice mayor a 1 coloca a la empresa en buena posición")

    # Estas razones son montos absolutos: se toma el valor absoluto
    cols_val = rf.columns[2:]
    for fila in ('Beneish M-Score', 'Gasto financiero(%)'):
        if fila in rf.index:
            rf.loc[fila, cols_val] = pd.to_numeric(rf.loc[fila, cols_val], errors='coerce').abs()
    return rf

## Calificación de ratios (0 a 1) con reglas por tipo de empresa

In [8]:
def _pendiente(series, idx, window=5):
    start = max(0, idx - window + 1)
    vals = series.iloc[start:idx + 1].dropna()
    if len(vals) < 2:
        return np.nan
    x = np.arange(len(vals), dtype=float)
    return np.polyfit(x, vals.values.astype(float), 1)[0]


def _score_tendencia(series, idx, bajista=False, window=5):
    s = _pendiente(series, idx, window)
    if pd.isna(s):
        return np.nan
    start = max(0, idx - window + 1)
    ref = abs(series.iloc[start:idx + 1].dropna().mean()) or 1
    norm = (s / ref) * (-1 if bajista else 1)
    if norm > 0.02:
        return 1.0
    if norm >= -0.02:
        return 0.5
    return 0.0


def calcular_score(ratio, val, series, idx, tipo, wacc_val=np.nan, rps=None):
    """Puntaje 0-1 de un ratio en un año. Las reglas dependen del tipo de empresa."""
    if pd.isna(val):
        return np.nan
    es_banco = tipo == TIPO_BANCO

    # ---------- Tendencia deseable (alcista) ----------
    tendencia_alcista = ('EPS', 'Revenue Per Share', 'Dividends per Share')
    if not es_banco:
        tendencia_alcista += ('Operating Margin(%)', 'Valor en libros', 'Free cash flow to sales(%)')
    if ratio in tendencia_alcista:
        if ratio == 'Dividends per Share':
            ventana = series.iloc[max(0, idx - 4):idx + 1].fillna(0)
            if (ventana == 0).all():
                return 0.0
        return _score_tendencia(series, idx)
    if ratio == '# Shares':
        return _score_tendencia(series, idx, bajista=True)

    # ---------- Ratios que en bancos se comparan con su propio historial ----------
    if es_banco and ratio in ('Free cash flow to sales(%)', 'Free Cash Flow to Debt ratio',
                              'net worth to long-term debt ratio', 'Gasto financiero(%)'):
        if isinstance(val, str) and val.lower() == 'no deuda':
            return 1.0
        hist = series.iloc[max(0, idx - 5):idx].dropna().mean()
        if pd.isna(hist) or hist == 0:
            return 0.5
        rel = val / hist
        if ratio == 'Gasto financiero(%)':   # menor es mejor
            return 1.0 if rel < 0.8 else (0.5 if rel <= 1.2 else 0.0)
        return 1.0 if rel >= 1.2 else (0.5 if rel >= 0.8 else 0.0)

    if isinstance(val, str) and val.lower() == 'no deuda':
        return 1.0 if ratio in ('Free Cash Flow to Debt ratio', 'net worth to long-term debt ratio',
                                'long-term debt-to-equity(%)') else np.nan

    # ---------- Rentabilidad ----------
    if ratio == 'Amortizacion(%)':
        return 1.0 if val < 10 else (0.5 if val <= 20 else 0.0)
    if ratio == 'Net Profit Margin(%)':
        if es_banco:
            if val > 25: return 1.0
            if val >= 15: return 0.75
            if val >= 5: return 0.25
            return 0.0
        if val > 20: return 1.0
        if val >= 10: return 0.5
        if val >= 0: return 0.25
        return 0.0
    if ratio == 'Gross Profit(%)':
        return 1.0 if val > 40 else (0.5 if val >= 20 else 0.0)
    if ratio == 'Inventory Turnover Ratio':
        return 1.0 if val > 8 else (0.5 if val >= 4 else 0.0)
    if ratio == 'Net Interest Margin (Bank Only) %':
        base = 1.0 if val >= 3.5 else (0.5 if val >= 2.5 else 0.0)
        if _score_tendencia(series, idx) == 0.0:
            base = min(base, 0.5)
        return base
    if ratio == 'ROA%':
        if es_banco:
            return 1.0 if val >= 1.5 else (0.75 if val >= 1.0 else (0.25 if val >= 0.5 else 0.0))
        if val > 20: return 1.0
        if val >= 7: return 0.5
        if val >= 0: return 0.25
        return 0.0
    if ratio == 'ROE%':
        if es_banco:
            if val >= 20: return 1.0
            if val >= 12: return 0.75
            if val >= 0: return 0.25
            return 0.0
        if val > 20: return 1.0
        if val >= 15: return 0.75
        if val >= 0: return 0.25
        return 0.0
    if ratio == 'Return-on-Tangible-Equity':
        if val > 15: return 1.0
        if val >= 10: return 0.75
        if val >= 5: return 0.25
        return 0.0
    if ratio == 'ROIC %':
        beat_wacc = (not pd.isna(wacc_val)) and val > wacc_val
        if beat_wacc and val > 15: return 1.0
        if beat_wacc: return 0.5
        return 0.0

    # ---------- Solvencia y apalancamiento ----------
    if ratio == 'Pasivo/Fondos Propios':
        if es_banco:
            if val <= 8: return 1.0
            if val <= 9.5: return 0.5
            return 0.0
        return 1.0 if val <= 1 else 0.0
    if ratio == 'long-term debt-to-equity(%)':
        if es_banco:
            if val < 0: return 0.0
            if val < 100: return 1.0
            if val < 150: return 0.5
            return 0.0
        if val < 0: return 0.0
        return 1.0 if val < 25 else (0.5 if val < 45 else 0.0)
    if ratio == 'ROB':
        if es_banco:
            return 1.0 if val < 10 else (0.5 if val < 11.5 else 0.0)
        return 1.0 if val < 4.5 else 0.0
    if ratio == 'Equity-to-Asset':
        return 1.0 if val >= 0.10 else (0.5 if val >= 0.06 else 0.0)
    if ratio == 'Efficiency Ratio(%)':
        if val < 50: return 1.0
        if val < 60: return 0.75
        if val < 70: return 0.25
        return 0.0
    if ratio == 'Loan to Deposit Ratio(%)':   # rango óptimo, no "mientras más bajo mejor"
        if 80 <= val <= 90: return 1.0
        if (70 <= val < 80) or (90 < val <= 100): return 0.5
        return 0.0
    if ratio == 'Free Cash Flow to Debt ratio':
        if val >= 1: return 1.0
        if val >= 0.8: return 0.75
        if val >= 0.6: return 0.25
        if val >= 0.4: return 0.10
        return 0.0
    if ratio == 'net worth to long-term debt ratio':
        if val >= 3: return 1.0
        if val >= 2: return 0.75
        if val >= 1: return 0.5
        if val >= 0.5: return 0.25
        return 0.0
    if ratio == 'Gasto financiero(%)':
        return 1.0 if val < 15 else (0.5 if val < 30 else 0.0)
    if ratio == 'Current Ratio':
        if val >= 1.5: return 1.0
        if val >= 1.0: return 0.75
        if val >= 0.5: return 0.25
        return 0.0
    if ratio == 'Inventory to current Assets(%)':
        return 1.0 if val < 40 else (0.25 if val < 60 else 0.0)
    if ratio == 'Receivable to Current Assets(%)':
        return 1.0 if val < 40 else (0.5 if val < 60 else 0.0)
    if ratio == 'Net Trading Cycle':
        if val < 0:
            rps_trend = _score_tendencia(rps, idx) if rps is not None else np.nan
            return 1.0 if rps_trend == 1.0 else 0.5
        trend = _score_tendencia(series, idx, bajista=True)
        return 0.0 if trend == 0.0 else 0.5

    # ---------- Dividendos y valoración ----------
    if ratio == 'Dividend Payout Ratio(%)':
        return 1.0 if 20 <= val <= 60 else 0.0
    if ratio == 'PEGY Ratio':
        if val == 0: return np.nan
        return 1.0 if val < 1 else (0.5 if val <= 1.75 else 0.0)
    if ratio == 'PER':
        if val <= 0: return np.nan
        if val < 9: return 1.0
        if val <= 25: return 0.75
        if val <= 40: return 0.25
        return 0.0
    if ratio == 'P/S Ratio':
        if val == 0: return np.nan
        hist = series.replace(0, np.nan).iloc[max(0, idx - 5):idx].mean()
        if pd.isna(hist) or hist == 0: return 0.5
        rel = val / hist
        return 1.0 if rel < 0.8 else (0.5 if rel <= 1.2 else 0.0)

    # ---------- Costo de capital ----------
    if ratio == 'WACC %':
        if es_banco:
            return 1.0 if val < 7 else (0.5 if val <= 12 else 0.0)
        return np.nan
    if ratio == 'WACC % Growth':
        return 1.0 if val <= 0 else (0.5 if val <= 15 else 0.0)

    # ---------- Calidad contable ----------
    if ratio == 'Piotrivski F-Score':
        return 1.0 if val >= 7 else (0.5 if val >= 4 else 0.0)
    if ratio == 'Altman Z score':
        return 1.0 if val > 3 else (0.5 if val >= 1.8 else 0.0)
    if ratio == 'Beneish M-Score':
        return 1.0 if val >= 1.78 else 0.0
    if ratio == 'Test de Acidez':
        return 1.0 if val >= 1 else (0.5 if val >= 0.5 else 0.0)

    # ---------- Aseguradoras ----------
    if ratio == 'Loss Ratio(%)':
        if ASEGURADORA_SALUD and val < 75: return 0.25
        if val < 60: return 1.0
        if val < 70: return 0.75
        if val < 80: return 0.5
        if val <= 90: return 0.25
        return 0.0
    if ratio == 'Expense Ratio(%)':
        if val < 12: return 1.0
        if val < 15: return 0.75
        if val < 20: return 0.5
        if val <= 25: return 0.25
        return 0.0
    if ratio == 'Investment Yield(%)':
        tasa = promedio_bonos_csv.get(anio_de(series.index[idx]), np.nan)
        if pd.isna(tasa): return np.nan
        spread = val - tasa
        if spread >= 1.5: return 1.0
        if spread >= 0.5: return 0.75
        if spread >= -0.5: return 0.5
        if spread >= -1.5: return 0.25
        return 0.0
    if ratio == 'Reserve-to-Premium Ratio':
        if idx == 0: return np.nan
        prev = pd.to_numeric(series.iloc[idx - 1], errors='coerce')
        if pd.isna(prev) or prev == 0: return np.nan
        cambio = (val - prev) / abs(prev) * 100
        if cambio < -15: return 0.0
        if cambio < -5: return 0.5
        if cambio <= 15: return 1.0
        if cambio <= 30: return 0.75
        return 0.5
    if ratio == 'Book Value Growth(%)':
        if val > 12: return 1.0
        if val >= 8: return 0.75
        if val >= 4: return 0.5
        if val >= 0: return 0.25
        return 0.0
    if ratio == 'Float Growth(%)':
        if val > 8: return 1.0
        if val >= 3: return 0.75
        if val >= 0: return 0.5
        if val >= -5: return 0.25
        return 0.0

    return np.nan


def calcular_scores(rf, tipo):
    """Puntaje por ratio y año, ponderado por importancia. Devuelve (tabla, score total por año)."""
    rf_r = rf.drop(index=[r for r in RATIOS_A_ELIMINAR if r in rf.index])
    year_cols = [c for c in rf_r.columns if str(c)[:3] in MES_MAP]

    wacc_row = (pd.to_numeric(rf_r.loc['WACC %', year_cols], errors='coerce')
                if 'WACC %' in rf_r.index else pd.Series(np.nan, index=year_cols))
    rps_row = (pd.to_numeric(rf_r.loc['Revenue Per Share', year_cols], errors='coerce')
               if 'Revenue Per Share' in rf_r.index else None)

    df_scores = pd.DataFrame(index=rf_r.index, columns=year_cols, dtype=float)
    for ratio in rf_r.index:
        raw = rf_r.loc[ratio, year_cols]
        vals = pd.to_numeric(raw, errors='coerce')
        for i, yr in enumerate(year_cols):
            v = raw.iloc[i]
            v = v if isinstance(v, str) and v.lower() == 'no deuda' else vals.iloc[i]
            df_scores.loc[ratio, yr] = calcular_score(ratio, v, vals, i, tipo, wacc_row.get(yr, np.nan), rps_row)
    df_scores = df_scores.astype(float)

    pesos = pd.Series(IMPORTANCIA[tipo]).reindex(df_scores.index).fillna(0)
    df_scores.insert(0, '% importancia', pesos)
    ponderados = df_scores[year_cols].multiply(pesos, axis=0)
    ponderados.insert(0, '% importancia', pesos)

    score_total = {}
    for yr in year_cols:
        col = df_scores[yr]
        mask = col.notna() & (pesos > 0)
        score_total[yr] = (np.nan if mask.sum() == 0 else
                           round((col[mask] * pesos[mask]).sum() / pesos[mask].sum() * 100, 1))

    fila_total = ponderados[year_cols].sum(skipna=True)
    fila_total["% importancia"] = pesos.sum()
    tabla = pd.concat([ponderados, fila_total.rename("Score Total").to_frame().T])
    return tabla, pd.Series(score_total, name='Score Total (%)')

## Valuaciones

Las valuaciones que aplican a todos los tipos se calculan siempre. Liquidación, EPV, NCAV y
WACC solo se calculan para no financieras (en bancos y aseguradoras no aplican igual).
Para bancos la base de utilidades es `Pre-Tax Income`; para los demás, `EBITDA`.

In [9]:
def _fecha_mes(etiquetas):
    return pd.to_datetime(etiquetas, format='%b%Y') + pd.offsets.MonthEnd(0)


def crecimiento_5_anios(df, columna):
    """Crecimiento promedio de los últimos 4 periodos (ventana de 5 puntos)."""
    df = df.copy()
    df['Fecha_Temp'] = _fecha_mes(df['Año'])
    df = df.sort_values('Fecha_Temp').reset_index(drop=True)
    resultados = []
    for i in range(len(df)):
        f_act = df.loc[i, 'Fecha_Temp']
        sub = df[(df['Fecha_Temp'] >= f_act - relativedelta(years=4)) & (df['Fecha_Temp'] <= f_act)]
        if len(sub) >= 2:
            pct = sub[columna].pct_change().replace([np.inf, -np.inf], np.nan).dropna()
            crec = pct.mean() if not pct.empty else 0
        else:
            crec = np.nan
        resultados.append({'Año': df.loc[i, 'Año'],
                           f'{columna}_crecimiento': round(crec, 3) if pd.notna(crec) else np.nan})
    return pd.DataFrame(resultados).dropna().reset_index(drop=True)


def cagr_5_anios(df, columna):
    """CAGR entre el punto de hace 4 años y el actual."""
    df = df.copy()
    df['Fecha_Temp'] = _fecha_mes(df['Año'])
    df = df.sort_values('Fecha_Temp').reset_index(drop=True)
    resultados = []
    for i in range(len(df)):
        f_fin = df.loc[i, 'Fecha_Temp']
        sub = df[(df['Fecha_Temp'] >= f_fin - relativedelta(years=4)) & (df['Fecha_Temp'] <= f_fin)]
        if len(sub) >= 2:
            v_ini = sub.iloc[0][columna]
            v_fin = sub.iloc[-1][columna]
            cagr = (v_fin / v_ini) ** (1 / 4) - 1 if (v_ini > 0 and v_fin > 0) else 0
        else:
            cagr = np.nan
        resultados.append({'Año': df.loc[i, 'Año'],
                           f'CAGR_{columna}': round(cagr, 3) if pd.notna(cagr) else np.nan})
    return pd.DataFrame(resultados).dropna().reset_index(drop=True)


def crecimiento_10_anios(df, columna):
    """Crecimiento promedio de hasta 10 periodos previos."""
    df = df.copy()
    df['Fecha_Temp'] = _fecha_mes(df['Año'])
    df = df.sort_values('Fecha_Temp').reset_index(drop=True)
    df[columna] = pd.to_numeric(df[columna], errors='coerce')
    resultados = []
    for i in range(1, len(df)):
        previos = df.loc[max(0, i - 10):i - 1, columna]
        if len(previos) >= 2:
            pct = previos.pct_change().replace([np.inf, -np.inf], np.nan).dropna()
            crec = pct.mean() if not pct.empty else 0
            resultados.append({'Año': df.loc[i, 'Año'], f'{columna}_crecimiento': round(float(crec), 3)})
    return pd.DataFrame(resultados)


def cagr_10_anios(df, columna, ventana_max=10, min_anios=2):
    """CAGR contra el primer valor positivo de hasta 10 periodos atrás."""
    df = df.copy()
    df['Fecha_Temp'] = _fecha_mes(df['Año'])
    df = df.sort_values('Fecha_Temp').reset_index(drop=True)
    resultados = []
    for i in range(1, len(df)):
        f_fin = df.loc[i, 'Fecha_Temp']
        v_fin = df.loc[i, columna]
        inicio = max(0, i - ventana_max)
        j = inicio
        while j < i and (pd.isna(df.loc[j, columna]) or df.loc[j, columna] <= 0):
            j += 1
        if (f_fin - df.loc[inicio, 'Fecha_Temp']).days / 365.25 < min_anios:
            continue
        if pd.isna(v_fin) or v_fin <= 0 or j >= i:
            cagr = 0
        else:
            n = (f_fin - df.loc[j, 'Fecha_Temp']).days / 365.25
            cagr = 0 if n < min_anios else (v_fin / df.loc[j, columna]) ** (1 / n) - 1
        resultados.append({'Año': df.loc[i, 'Año'], f'CAGR_{columna}': round(float(cagr), 3)})
    return pd.DataFrame(resultados)


def seleccionar_crecimiento(valores):
    """Crecimiento a usar en el modelo de crecimiento (entre 5% y 11%)."""
    if not valores:
        return 0.05
    if all(v < 0.05 for v in valores):
        return 0.05
    if all(v > 0.11 for v in valores):
        return 0.11
    en_rango = [v for v in valores if 0.05 <= v <= 0.11]
    if en_rango:
        return min(en_rango)
    promedio = sum(valores) / len(valores)
    return min(max(promedio, 0.05), 0.11)


def valor_intrinseco_dcf(x_df, y_df, base_df, col_base, nombre):
    """Flujo descontado de 10 años + valor terminal: f * (x(1-x^10)/(1-x) + x^10*y(1-y^10)/(1-y))."""
    m = base_df[['Año', col_base]].merge(x_df, on='Año').merge(y_df, on='Año')
    x, y, f = m['x'], m['y'], m[col_base]
    term1 = x * (1 - x ** 10) / (1 - x)
    term2 = (x ** 10) * y * (1 - y ** 10) / (1 - y)
    m[nombre] = f * (term1 + term2)
    return m[['Año', nombre]]


def crecimiento_ponderado_10(crec_serie, pesos):
    """Crecimiento ponderado por el flujo del año anterior, ventana móvil de 10 años."""
    comunes = [i for i in crec_serie.index if i in pesos.index]
    c = crec_serie.reindex(comunes)
    w = pesos.reindex(comunes)
    return (c * w).rolling(10, min_periods=1).sum() / w.rolling(10, min_periods=1).sum()


def a_df_con_ano(serie, nombre):
    return pd.DataFrame({'Año': serie.index.astype(str), nombre: serie.values}).dropna()


def calcular_valuaciones(per, mkt, rf, tipo):
    df = per['df_anual_filtrado']
    dfq = per['df_trimestral_filtrado']
    headers, last_20, last_20_mes = per['valid_headers'], per['last_20_years'], per['last_20_years_mes']
    fila_ebitda = 'Pre-Tax Income' if tipo == TIPO_BANCO else 'EBITDA'
    piezas = {}   # columna de valuación -> DataFrame (Año, columna)

    acciones = pd.to_numeric(serie_fila(df, 'Shares Outstanding (Basic Average)'), errors='coerce')
    activos = pd.to_numeric(serie_fila(df, 'Total Assets'), errors='coerce')
    pasivos = pd.to_numeric(serie_fila(df, 'Total Liabilities'), errors='coerce')
    patrimonio = pd.to_numeric(serie_fila(df, 'Total Stockholders Equity'), errors='coerce')
    preferente = pd.to_numeric(serie_fila(df, 'Preferred Stock'), errors='coerce')
    intangibles = pd.to_numeric(serie_fila(df, 'Intangible Assets'), errors='coerce')

    # ---- Book value por acción ----
    bvps = (activos - pasivos) / acciones
    piezas['Book_Value_per_Share'] = serie_a_df_anual(bvps, 'Book_Value_per_Share')

    # ---- Tangible book value por acción ----
    tbv = (patrimonio - preferente - intangibles).round(2)
    tbvps = (tbv / acciones).round(2)
    piezas['Tangible_Book_Value_Per_Share'] = serie_a_df_anual(tbvps, 'Tangible_Book_Value_Per_Share')

    # ---- Graham Number ----
    eps = pd.to_numeric(serie_fila(df, 'EPS without NRI'), errors='coerce')
    df_eps = pd.DataFrame({'Año': eps.index.astype(str), 'EPS': eps.values})
    g = piezas['Tangible_Book_Value_Per_Share'].merge(df_eps, on='Año', how='inner')
    g['Graham_Number'] = np.where(g['EPS'] > 0,
                                  np.sqrt(22.5 * g['Tangible_Book_Value_Per_Share'] * g['EPS']), 0)
    piezas['Graham_Number'] = g[['Año', 'Graham_Number']]

    # ---- DCF por crecimiento (modelo de crecimiento + flujo ajustado por inflación) ----
    ebitda = serie_fila(df, fila_ebitda)
    fcf = serie_fila(df, 'Free Cash Flow')
    revenue = serie_fila(df, 'Revenue')
    df_ebitda = pd.DataFrame({'Año': ebitda.index.astype(str), 'EBITDA': pd.to_numeric(ebitda.values, errors='coerce')})
    df_fcf = pd.DataFrame({'Año': fcf.index.astype(str), 'Free_Cash_Flow': pd.to_numeric(fcf.values, errors='coerce')})
    df_rev = pd.DataFrame({'Año': revenue.index.astype(str), 'Revenue': pd.to_numeric(revenue.values, errors='coerce')})

    crec5 = (crecimiento_5_anios(df_ebitda, 'EBITDA')
             .merge(crecimiento_5_anios(df_fcf, 'Free_Cash_Flow'), on='Año')
             .merge(crecimiento_5_anios(df_rev, 'Revenue'), on='Año'))
    cagr5 = (cagr_5_anios(df_ebitda, 'EBITDA')
             .merge(cagr_5_anios(df_fcf, 'Free_Cash_Flow'), on='Año')
             .merge(cagr_5_anios(df_rev, 'Revenue'), on='Año'))
    crec_final = crec5.merge(cagr5, on='Año')
    columnas_crec = [c for c in crec_final.columns if c != 'Año']
    crec_final['Crecimiento_Seleccionado'] = [
        round(seleccionar_crecimiento([row[c] for c in columnas_crec
                                       if isinstance(row[c], (int, float)) and pd.notna(row[c])]), 3)
        for _, row in crec_final.iterrows()
    ]
    crec_final['Growth_Multiple'] = 8.3459 * (1.07 ** (crec_final['Crecimiento_Seleccionado'] * 100 - 4))
    crec_final['Growth_Multiple'] = crec_final['Growth_Multiple'].round(2)

    # Promedio de FCF de 5 años y promedio de 4 trimestres
    promedios_fcf = []
    fcf_vals = df_fcf.reset_index(drop=True)
    for i in range(4, len(fcf_vals)):
        promedios_fcf.append({'Año': fcf_vals.loc[i, 'Año'],
                              'FCF_5y_Promedio': round(fcf_vals.loc[i - 4:i, 'Free_Cash_Flow'].mean(), 2)})
    df_fcf_5y = pd.DataFrame(promedios_fcf)

    fcf_t = serie_fila(dfq, 'Free Cash Flow')
    df_fcf_trim = pd.DataFrame({'Periodo': fcf_t.index.astype(str),
                                'FCF_4T': pd.to_numeric(fcf_t.values, errors='coerce')})
    df_fcf_trim['Fecha_Temp'] = _fecha_mes(df_fcf_trim['Periodo'])
    df_fcf_trim = df_fcf_trim.sort_values('Fecha_Temp').reset_index(drop=True)
    res_trim = []
    for _, r in df_fcf_5y.iterrows():
        pos = df_fcf_trim.index[df_fcf_trim['Periodo'] == r['Año']]
        prom = None
        if len(pos) and pos[0] >= 3:
            i = pos[0]
            prom = round(df_fcf_trim.loc[i - 3:i, 'FCF_4T'].mean(), 2)
        res_trim.append({'Año': r['Año'], 'FCF_Trimestral_Promedio': prom})
    df_fcf_4t = pd.DataFrame(res_trim).dropna().reset_index(drop=True)

    fcf_aj = df_fcf_5y.merge(df_fcf_4t, on='Año', how='inner')
    fcf_aj['FCF_Ajustado'] = ((6 * fcf_aj['FCF_5y_Promedio'] + 0.75 * fcf_aj['FCF_Trimestral_Promedio']) / 6.75).round(2)
    fcf_aj['Año_Num'] = pd.to_datetime(fcf_aj['Año'], format='%b%Y').dt.year
    factor = dict(zip(mkt['inflacion']['Año'], mkt['inflacion']['Factor_Inflacion']))
    fcf_aj['Factor_Inflacion'] = fcf_aj['Año_Num'].map(factor)
    fcf_aj = fcf_aj.dropna(subset=['Factor_Inflacion'])
    fcf_aj['FCF_Ajustado_Inflacion'] = (fcf_aj['FCF_Ajustado'] * fcf_aj['Factor_Inflacion']).round(2)

    patrimonio_df = pd.DataFrame({'Año': patrimonio.index.astype(str),
                                  'Total_Stockholders_Equity': patrimonio.values})
    acciones_df = pd.DataFrame({'Año': acciones.index.astype(str), 'Total_Shares': acciones.values})
    intr = (crec_final[['Año', 'Growth_Multiple']]
            .merge(fcf_aj[['Año', 'FCF_Ajustado_Inflacion']], on='Año', how='inner')
            .merge(patrimonio_df, on='Año', how='inner')
            .merge(acciones_df, on='Año', how='inner'))
    intr['Intrinsic_Value'] = ((intr['Growth_Multiple'] * intr['FCF_Ajustado_Inflacion']
                                + 0.7 * intr['Total_Stockholders_Equity']) / intr['Total_Shares']).round(2)
    piezas['Intrinsic_Value'] = intr[['Año', 'Intrinsic_Value']]

    # ---- Mediana de P/S (precio promedio y mediano de 12 meses sobre ventas por acción) ----
    rev_s = pd.to_numeric(serie_fila(df, 'Revenue'), errors='coerce')
    rev_s.index = rev_s.index.astype(str)
    acc_s = acciones.copy()
    acc_s.index = acc_s.index.astype(str)
    rps = (rev_s / acc_s).dropna()

    precios = mkt['precios']

    def cierre_12m(etiqueta):
        fin = fecha_fin_de_mes(etiqueta)
        ini = fin - pd.DateOffset(years=1) + pd.DateOffset(days=1)
        c = precios.loc[(precios['Fecha'] >= ini) & (precios['Fecha'] <= fin), 'Cierre']
        return (c.mean(), c.median()) if len(c) else (np.nan, np.nan)

    pares = {lab: cierre_12m(lab) for lab in rps.index}
    prom_c = pd.Series({k: v[0] for k, v in pares.items()})
    med_c = pd.Series({k: v[1] for k, v in pares.items()})
    ratio_prom = ordenar_por_fecha((prom_c / rps).dropna())
    ratio_med = ordenar_por_fecha((med_c / rps).dropna())
    mm_prom = ratio_prom.rolling(10, min_periods=1).median().dropna()
    mm_med = ratio_med.rolling(10, min_periods=1).median().dropna()
    df_ps = pd.DataFrame({'Mediana_Prom': mm_prom, 'Mediana_Med': mm_med, 'RPS': rps}).dropna()
    piezas['Median_ps_value_promedio'] = pd.DataFrame(
        {'Año': df_ps.index.astype(str), 'Median_ps_value_promedio': (df_ps['Mediana_Prom'] * df_ps['RPS']).values})
    piezas['Median_ps_value_promedio_mediana'] = pd.DataFrame(
        {'Año': df_ps.index.astype(str), 'Median_ps_value_promedio_mediana': (df_ps['Mediana_Med'] * df_ps['RPS']).values})

    # ---- Peter Lynch: crecimiento de 5 años de la base de utilidades x EPS (x PEG en la versión 2) ----
    peg = pd.to_numeric(serie_fila(df, 'PEG Ratio'), errors='coerce')
    base_util = ordenar_por_fecha(pd.to_numeric(serie_fila(df, fila_ebitda), errors='coerce'))
    crec_pl = (base_util.pct_change().rolling(window=5, min_periods=1).mean().round(3) * 100).dropna()
    df_pl = pd.DataFrame({'Año': crec_pl.index.astype(str), 'Crecimiento': crec_pl.values})
    df_pl['EPS'] = df_pl['Año'].map(dict(zip(eps.index.astype(str), eps.values)))
    df_pl['PEG'] = df_pl['Año'].map(dict(zip(peg.index.astype(str), peg.values)))
    df_pl['Peter_Lynch_Value_1'] = 1 * df_pl['Crecimiento'] * df_pl['EPS']
    df_pl['Peter_Lynch_Value_2'] = df_pl['PEG'] * df_pl['Crecimiento'] * df_pl['EPS']
    piezas['Peter_Lynch_Value_1'] = df_pl[['Año', 'Peter_Lynch_Value_1']]
    piezas['Peter_Lynch_Value_2'] = df_pl[['Año', 'Peter_Lynch_Value_2']]

    # ---- DCF por flujo de caja libre ----
    tasa = mkt['tasa'].copy()
    tasa['Discount_Rate'] = tasa['Tasa_Libre_Riesgo'] + MARKET_RISK_PREMIUM
    tasa_desc = tasa[['Año', 'Discount_Rate']]
    y_df = pd.DataFrame({'Año': tasa_desc['Año'],
                         'y': (1 + TASA_CRECIMIENTO_DCF) / (1 + tasa_desc['Discount_Rate'])})

    fcf_ps = ordenar_por_fecha(pd.to_numeric(serie_fila(df, 'Free Cash Flow per Share'), errors='coerce'))
    crec_fcf_ps = fcf_ps.pct_change().dropna()
    fcf_total = ordenar_por_fecha(pd.to_numeric(fcf, errors='coerce'))
    pesos_fcf = fcf_total.shift(1)

    def como_x(serie_crec, nombre_col):
        d = pd.DataFrame({'Año': serie_crec.index.astype(str), nombre_col: serie_crec.values})
        return d

    w_fcf = crecimiento_ponderado_10(crec_fcf_ps, pesos_fcf).clip(0.05, 0.25)
    med_fcf = crec_fcf_ps.rolling(10, min_periods=1).median().clip(0.05, 0.25)
    avg_fcf = crecimiento_10_anios(df_fcf, 'Free_Cash_Flow').copy()
    avg_fcf['Free_Cash_Flow_crecimiento'] = avg_fcf['Free_Cash_Flow_crecimiento'].clip(0.05, 0.25)
    cagr_fcf = cagr_10_anios(df_fcf, 'Free_Cash_Flow').copy()
    cagr_fcf['CAGR_Free_Cash_Flow'] = cagr_fcf['CAGR_Free_Cash_Flow'].clip(0.05, 0.25)

    def x_de(df_g, col):
        m = df_g.merge(tasa_desc, on='Año', how='inner')
        m['x'] = (1 + m[col]) / (1 + m['Discount_Rate'])
        return m[['Año', 'x']]

    fcf_ps_df = a_df_con_ano(fcf_ps, 'FCF_per_Share')
    piezas['Intrinsic_Weighted'] = valor_intrinseco_dcf(
        x_de(como_x(w_fcf, 'Weighted'), 'Weighted'), y_df, fcf_ps_df, 'FCF_per_Share', 'Intrinsic_Weighted')
    piezas['Intrinsic_Mediana'] = valor_intrinseco_dcf(
        x_de(como_x(med_fcf, 'Mediana'), 'Mediana'), y_df, fcf_ps_df, 'FCF_per_Share', 'Intrinsic_Mediana')
    piezas['Intrinsic_FCF_Avg'] = valor_intrinseco_dcf(
        x_de(avg_fcf.rename(columns={'Free_Cash_Flow_crecimiento': 'g'}), 'g'), y_df, fcf_ps_df,
        'FCF_per_Share', 'Intrinsic_FCF_Avg')
    piezas['Intrinsic_CAGR_FCF'] = valor_intrinseco_dcf(
        x_de(cagr_fcf.rename(columns={'CAGR_Free_Cash_Flow': 'g'}), 'g'), y_df, fcf_ps_df,
        'FCF_per_Share', 'Intrinsic_CAGR_FCF')

    # ---- DCF por utilidades (EPS sin partidas no recurrentes) ----
    eps_s = ordenar_por_fecha(eps)
    crec_eps = eps_s.pct_change().dropna()
    w_eps = crecimiento_ponderado_10(crec_eps, eps_s.shift(1)).clip(0.05, 0.20)
    med_eps = crec_eps.rolling(10, min_periods=1).median().clip(0.05, 0.20)
    avg_eps = crecimiento_10_anios(df_eps, 'EPS').copy()
    avg_eps['EPS_crecimiento'] = avg_eps['EPS_crecimiento'].clip(0.05, 0.20)
    cagr_eps = cagr_10_anios(df_eps, 'EPS').copy()
    cagr_eps['CAGR_EPS'] = cagr_eps['CAGR_EPS'].clip(0.05, 0.20)
    eps_df = df_eps.dropna(subset=['EPS'])
    piezas['Intrinsic_EPS_Weighted'] = valor_intrinseco_dcf(
        x_de(como_x(w_eps, 'Weighted_eps'), 'Weighted_eps'), y_df, eps_df, 'EPS', 'Intrinsic_EPS_Weighted')
    piezas['Intrinsic_EPS_Mediana'] = valor_intrinseco_dcf(
        x_de(como_x(med_eps, 'Mediana_eps'), 'Mediana_eps'), y_df, eps_df, 'EPS', 'Intrinsic_EPS_Mediana')
    piezas['Intrinsic_EPS_Avg'] = valor_intrinseco_dcf(
        x_de(avg_eps.rename(columns={'EPS_crecimiento': 'g'}), 'g'), y_df, eps_df, 'EPS', 'Intrinsic_EPS_Avg')
    piezas['Intrinsic_CAGR_EPS'] = valor_intrinseco_dcf(
        x_de(cagr_eps.rename(columns={'CAGR_EPS': 'g'}), 'g'), y_df, eps_df, 'EPS', 'Intrinsic_CAGR_EPS')

    # ---------- Solo no financieras ----------
    if tipo == TIPO_NO_FIN:
        def fila_num(nombre, factor=1.0):
            return pd.to_numeric(serie_fila(df, nombre), errors='coerce') * factor

        # Valor de liquidación
        liq = (fila_num('Cash, Cash Equivalents, Marketable Securities', 0.95)
               + fila_num('Total Receivables', 0.75)
               + fila_num('Total Inventories', 0.60)
               + fila_num('Property, Plant and Equipment', 0.70)
               + pd.to_numeric(serie_fila(df, 'Investments And Advances'), errors='coerce')
               + fila_num('Other Long Term Assets', 0.30)
               - pasivos)
        piezas['Liquidation_Value_Per_Share'] = serie_a_df_anual(liq / acciones, 'Liquidation_Value_Per_Share')

        # NCAV: activo circulante - pasivos - minoritario - preferentes
        minoritario = pd.to_numeric(serie_fila(df, 'Minority Interest'), errors='coerce')
        ncav = (pd.to_numeric(serie_fila(df, 'Total Current Assets'), errors='coerce')
                - pasivos - minoritario - preferente).round(2)
        piezas['NCAV_Per_Share'] = serie_a_df_anual((ncav / acciones).round(2), 'NCAV_Per_Share')

        # EPV: utilidad normalizada - mantenimiento de capex, descontada a WACC
        def promedio_previo(serie_ord, ventana=5):
            return serie_ord.shift(1).rolling(ventana, min_periods=1).mean()

        margen = promedio_previo(ordenar_por_fecha(pd.to_numeric(rf.loc['Operating Margin(%)'].iloc[2:], errors='coerce'))).round(2) / 100
        ventas_prom = promedio_previo(ordenar_por_fecha(pd.to_numeric(rf.loc['Sales'].iloc[2:], errors='coerce'))).round(2)

        sga_q = ordenar_por_fecha(pd.to_numeric(serie_fila(dfq, 'Selling, General, & Admin. Expense'), errors='coerce'))
        fechas_q = pd.to_datetime(pd.Index(sga_q.index).astype(str), format='%b%Y')
        sga_prom = {}
        for lab in last_20_mes:
            corte = pd.to_datetime(lab, format='%b%Y') + pd.DateOffset(months=1)
            previos = sga_q[np.asarray(fechas_q < corte)].tail(20)
            sga_prom[lab] = round((previos * 0.75).mean(), 2) if len(previos) else 0
        sga_prom = pd.Series(sga_prom)

        ebit = pd.DataFrame({'ventas': ventas_prom, 'margen': margen, 'sga': sga_prom}).dropna()
        ebit_norm = (ebit['ventas'] * ebit['margen'] + ebit['sga']).round(2)

        tax_anual = pd.Series(pd.to_numeric(serie_fila(df, 'Tax Rate %'), errors='coerce').values,
                              index=[anio_de(i) for i in serie_fila(df, 'Tax Rate %').index])
        tax_prom = {}
        for a in last_20:
            ult = tax_anual[tax_anual.index < a].tail(5)
            tax_prom[a] = round(ult[ult >= 0].mean(), 2) / 100
        tax_prom = pd.Series(tax_prom).fillna(0)

        dda_anual = pd.Series(pd.to_numeric(serie_fila(df, 'Depreciation, Depletion and Amortization'), errors='coerce').values,
                              index=[anio_de(i) for i in serie_fila(df, 'Depreciation, Depletion and Amortization').index])
        dda_prom = pd.Series({a: round(dda_anual[dda_anual.index < a].tail(5).mean(), 2) for a in last_20})
        dep_exceso = dda_prom * 0.5 * tax_prom

        anios_ebit = pd.Series([anio_de(i) for i in ebit.index], index=ebit.index)
        ebit_aj = ebit_norm * (1 - anios_ebit.map(tax_prom))
        ne = ebit_aj + anios_ebit.map(dep_exceso)     # utilidad normalizada por periodo
        ne = ordenar_por_fecha(ne.dropna())

        acum = serie_fila(df, '  Accumulated Depreciation').copy()
        if acum.iloc[-1] == 0:
            acum.iloc[-1] = acum.iloc[-2]
        acum = pd.to_numeric(acum, errors='coerce').abs()
        bruto = pd.to_numeric(serie_fila(df, 'Gross Property, Plant and Equipment'), errors='coerce')
        net_ppe = (bruto - acum).reindex(last_20_mes)

        capex = pd.to_numeric(serie_fila(df, 'Capital Expenditure'), errors='coerce').abs()
        rev_capex = pd.to_numeric(serie_fila(df, 'Revenue'), errors='coerce')
        crec_rev = rev_capex.pct_change()
        prom_crec = {}
        for lab in last_20_mes:
            if lab not in crec_rev.index:
                prom_crec[lab] = np.nan
                continue
            p = list(crec_rev.index).index(lab)
            prom_crec[lab] = round(crec_rev.iloc[max(0, p - 5):p].mean(), 4)
        df_in = pd.DataFrame({'Capex': capex, 'Revenue': rev_capex, 'Net_PPE': net_ppe,
                              'Prom_Crec': pd.Series(prom_crec)}).dropna(subset=['Capex', 'Revenue', 'Net_PPE'])
        growth_capex = ((df_in['Net_PPE'] / df_in['Revenue']) * (df_in['Prom_Crec'] * df_in['Revenue']))
        growth_capex = growth_capex.where(df_in['Prom_Crec'] > 0, 0)
        mant = df_in['Capex'] - growth_capex
        mant = mant.where((df_in['Prom_Crec'] > 0) & (mant >= 0), df_in['Capex'])

        # WACC: capital propio a precio de mercado y deuda a costo de intereses
        cierres = []
        for lab in acciones.index:
            ref = fecha_fin_de_mes(lab)
            c = precios.loc[(precios['Fecha'] > ref - relativedelta(years=1)) & (precios['Fecha'] <= ref), 'Cierre']
            cierres.append((c.mean(), c.median()))
        precio_min = pd.DataFrame(cierres, index=acciones.index.astype(str)).min(axis=1)
        E = (acciones.set_axis(acciones.index.astype(str)) * precio_min).round(2)

        deuda = (pd.to_numeric(serie_fila(df, 'Short-Term Debt & Capital Lease Obligation'), errors='coerce')
                 + pd.to_numeric(serie_fila(df, 'Long-Term Debt & Capital Lease Obligation'), errors='coerce'))
        deuda.index = deuda.index.astype(str)
        intereses = pd.to_numeric(serie_fila(df, '  Interest Expense'), errors='coerce').abs()
        intereses.index = intereses.index.astype(str)
        rd = (intereses / deuda).round(5)
        t_imp = pd.to_numeric(serie_fila(df, 'Tax Rate %'), errors='coerce') / 100
        t_imp.index = t_imp.index.astype(str)
        beta_s = mkt['beta'].set_index('Año')['Beta'] if len(mkt['beta']) else pd.Series(dtype=float)
        tasa_s = mkt['tasa'].set_index('Año')['Tasa_Libre_Riesgo']
        re_s = (tasa_s + beta_s * MARKET_RISK_PREMIUM).round(5)

        wacc_df = pd.DataFrame({'E': E, 'D': deuda, 're': re_s, 'rd': rd, 't': t_imp}).dropna()
        wacc_df['WACC'] = ((wacc_df['E'] / (wacc_df['E'] + wacc_df['D'])) * wacc_df['re']
                           + (wacc_df['D'] / (wacc_df['E'] + wacc_df['D'])) * wacc_df['rd'] * (1 - wacc_df['t']))
        wacc_s = wacc_df['WACC'].round(5)

        base_epv = pd.DataFrame({'NE': pd.Series(ne.values, index=ne.index.astype(str)),
                                 'MC': mant.set_axis(mant.index.astype(str)),
                                 'WACC': wacc_s}).dropna()
        base_epv = base_epv[base_epv['WACC'] != 0]
        epv_ops = ((base_epv['NE'] - base_epv['MC']) / base_epv['WACC']).round(2)

        caja = pd.to_numeric(serie_fila(df, 'Cash, Cash Equivalents, Marketable Securities'), errors='coerce')
        caja.index = caja.index.astype(str)
        acc_str = acciones.copy()
        acc_str.index = acc_str.index.astype(str)
        epv_final = ((epv_ops + caja - deuda) / acc_str).round(2)
        piezas['EPV_Final_Por_Accion'] = serie_a_df_anual(epv_final, 'EPV_Final_Por_Accion')

    # ---- Valores de GuruFocus (referencia) ----
    guru = [
        ('Book Value per Share', 'Book_Value_per_Share_Guru'),
        ('Tangible Book per Share', 'Tangible_Book_Value_per_Share_Guru'),
        ('Intrinsic Value: Projected FCF', 'Projected_FCF_Guru'),
        ('Median PS Value', 'Median_PS_Value_Guru'),
        ('Peter Lynch Fair Value', 'Peter_Lynch_Fair_Value_Guru'),
        ('Graham Number', 'Graham_Number_Guru'),
        ('Earnings Power Value (EPV)', 'Earnings_Power_Value_Guru'),
        ('GF Value', 'GF_Value_Guru'),
    ]
    for fila, nombre in guru:
        if (df['Fiscal Period'] == fila).any():
            serie = serie_fila(df, fila)
        else:
            serie = pd.Series(0, index=df.columns[1:])
        piezas[nombre] = serie_a_df_anual(serie, nombre)

    # ---- Unión de todas las valuaciones por periodo ----
    dfs = [d for d in piezas.values() if not d.empty]
    val = reduce(lambda a, b: a.merge(b, on='Año', how='outer'), dfs)
    val = (val.assign(_f=pd.to_datetime(val['Año'], format='%b%Y', errors='coerce'))
           .sort_values('_f').drop(columns='_f').reset_index(drop=True))
    if len(val) > 5:   # se quitan los primeros 5 periodos, como en los notebooks originales
        val = val.iloc[5:].reset_index(drop=True)
    return val

## Modelo de targets

Para cada valuación se revisa si el precio toca la banda de ±5% dentro de 156 semanas.
Se conserva lo que cumple hit rate ≥ 70% en el total **y** en los últimos 10 años.

In [10]:
def primer_toque(fecha_base, fecha_fin, valor, L, U, prices):
    """Primera semana en que el precio cruza la banda [L, U] dentro de la ventana."""
    if valor is None or pd.isna(valor) or valor <= 0:
        return np.nan, np.nan, pd.NaT, pd.NA

    w = prices.loc[(prices['Fecha'] >= fecha_base) & (prices['Fecha'] <= fecha_fin),
                   ['Fecha', 'Máximo', 'Mínimo']]
    if w.empty:
        return False, np.nan, pd.NaT, pd.NA

    hit_mask = (w['Mínimo'] <= U) & (w['Máximo'] >= L)
    p_base = (w.iloc[0]['Máximo'] + w.iloc[0]['Mínimo']) / 2

    def lado(p):
        if p < L: return "long"
        if p > U: return "short"
        return "at_target"

    if not hit_mask.any():
        return False, np.nan, pd.NaT, lado(p_base)

    pos = int(np.argmax(hit_mask.to_numpy()))
    hit_date = w.iloc[pos]['Fecha']
    side = "at_target" if pos == 0 else lado(p_base)
    return True, pos, hit_date, side


def calcular_objetivos(per, val, precios):
    headers_df = pd.DataFrame({'Año': per['valid_headers']})
    fb = pd.to_datetime(headers_df['Año'], format='%b%Y')
    headers_df['Fecha_teorica'] = fb + pd.DateOffset(months=1)

    valuations = val.merge(headers_df[['Año', 'Fecha_teorica']], on='Año', how='left').dropna(subset=['Fecha_teorica'])
    precios = precios.copy()
    precios[['Máximo', 'Mínimo']] = precios[['Máximo', 'Mínimo']].apply(
        lambda s: pd.to_numeric(s.astype(str).str.replace(',', '', regex=False).str.replace(' ', '', regex=False),
                                errors='coerce'))
    precios = precios.dropna(subset=['Fecha', 'Máximo', 'Mínimo']).sort_values('Fecha').reset_index(drop=True)

    valuations = pd.merge_asof(
        valuations.sort_values('Fecha_teorica'),
        precios[['Fecha']].sort_values('Fecha'),
        left_on='Fecha_teorica', right_on='Fecha', direction='forward'
    ).rename(columns={'Fecha': 'Fecha_base'})

    valuations = valuations.loc[:, ~valuations.columns.duplicated()].copy()
    id_cols = ['Año', 'Fecha_base']
    metodos = [c for c in valuations.columns
               if c not in id_cols and pd.api.types.is_numeric_dtype(valuations[c])]
    long = valuations.melt(id_vars=id_cols, value_vars=metodos, var_name='Metodo', value_name='Valor')
    long['Valor'] = pd.to_numeric(long['Valor'], errors='coerce')
    long = long.dropna(subset=['Valor']).reset_index(drop=True)

    long['L'] = long['Valor'] * (1 - TOL_OBJETIVO)
    long['U'] = long['Valor'] * (1 + TOL_OBJETIVO)
    long['Fecha_fin'] = long['Fecha_base'] + pd.to_timedelta(H_SEMANAS, unit='W')

    res = long.apply(
        lambda r: primer_toque(r['Fecha_base'], r['Fecha_fin'], r['Valor'], r['L'], r['U'], precios),
        axis=1, result_type='expand')
    res.columns = ['hit', 'tth_weeks', 'hit_date', 'side']
    long[['hit', 'tth_weeks', 'hit_date', 'side']] = res
    long['hit'] = long['hit'].replace({pd.NaT: np.nan}).astype('boolean')

    summary = (long.groupby('Metodo', as_index=False)
               .agg(targets=('hit', 'count'), hits=('hit', 'sum'), hit_rate=('hit', 'mean'),
                    median_tth=('tth_weeks', 'median'), avg_tth=('tth_weeks', 'mean')))
    for años, semanas in [(1, 52), (2, 104), (3, 156)]:
        tmp = long['hit'] & (long['tth_weeks'] <= semanas)
        summary[f'hit_le_{años}y'] = summary['Metodo'].map(long.assign(_t=tmp).groupby('Metodo')['_t'].mean())
    summary = summary.sort_values(['hit_rate', 'median_tth'], ascending=[False, True]).reset_index(drop=True)

    # Hit rate de los últimos 10 años por método
    fecha_anio = pd.to_datetime(long['Año'], format='%b%Y', errors='coerce')
    hace_10 = fecha_anio.max() - pd.DateOffset(years=10)
    hit_10y = long[fecha_anio >= hace_10].groupby('Metodo')['hit'].mean()
    cumple = summary['Metodo'].map(hit_10y).fillna(0) >= 0.70
    if EXIGIR_HIT_GLOBAL:
        cumple = cumple & (summary['hit_rate'] >= 0.70)
    metodos_top = summary.loc[cumple, 'Metodo']

    df_hits = (long.loc[long['Metodo'].isin(metodos_top),
                        ['Año', 'Metodo', 'Valor', 'hit', 'tth_weeks', 'hit_date', 'side']]
               .assign(_fecha=lambda d: pd.to_datetime(d['Año'], format='%b%Y', errors='coerce'))
               .sort_values(['Metodo', '_fecha']).drop(columns='_fecha').reset_index(drop=True))
    return df_hits

## Proceso por empresa y ciclo principal

In [ ]:
def exportar(ticket, rf, tabla_scores, df_hits):
    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    tabla_scores.to_excel(f"{CARPETA_SALIDA}/{ticket}_score_ponderados.xlsx")
    rf.to_excel(f"{CARPETA_SALIDA}/{ticket}_razones_financieras.xlsx")
    df_hits.to_excel(f"{CARPETA_SALIDA}/{ticket}_valuaciones_hits_por_anio.xlsx", index=False)


def procesar_ticket(ticket):
    df_err = cargar_excel(ticket)
    tipo = detectar_tipo(df_err)
    per = preparar_periodos(df_err)
    mkt = cargar_mercado(ticket, per)
    rf = calcular_razones(per['df_anual_filtrado'], tipo)
    val = calcular_valuaciones(per, mkt, rf, tipo)
    df_hits = calcular_objetivos(per, val, mkt['precios'])
    tabla_scores, score_total = calcular_scores(rf, tipo)
    exportar(ticket, rf, tabla_scores, df_hits)
    ultimo = score_total.dropna()
    return {
        'Ticket': ticket,
        'Tipo': tipo,
        'Score total último año (%)': ultimo.iloc[-1] if len(ultimo) else np.nan,
        'Métodos que cumplen': df_hits['Metodo'].nunique(),
    }


resultados, errores = [], []
for ticket in TICKETS:
    try:
        resultados.append(procesar_ticket(ticket))
        print(f"[{ticket}] OK")
    except Exception as e:
        errores.append({'Ticket': ticket, 'Error': f"{type(e).__name__}: {e}"})
        print(f"[{ticket}] ERROR: {type(e).__name__}: {e}")
        traceback.print_exc()

resumen = pd.DataFrame(resultados)
errores_df = pd.DataFrame(errores, columns=['Ticket', 'Error'])
print(resumen.to_string(index=False) if len(resumen) else "Sin resultados")
print("\nErrores:")
print(errores_df.to_string(index=False) if len(errores_df) else "Ninguno")

c:\Users\mar_c\OneDrive\Documentos\GuruFocus\guru_focus\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


[ABG] OK


C:\Users\mar_c\AppData\Local\Temp\ipykernel_27444\769446708.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('-', np.nan, inplace=True)
C:\Users\mar_c\AppData\Local\Temp\ipykernel_27444\769446708.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('-', np.nan, inplace=True)
C:\Users\mar_c\AppData\Local\Temp\ipykernel_27444\903112696.py:94: RuntimeWarning: divide by zero encountered in divide
  r = n / d


[ACN] OK


C:\Users\mar_c\AppData\Local\Temp\ipykernel_27444\769446708.py:37: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('-', np.nan, inplace=True)
C:\Users\mar_c\AppData\Local\Temp\ipykernel_27444\903112696.py:94: RuntimeWarning: divide by zero encountered in divide
  r = n / d
c:\Users\mar_c\OneDrive\Documentos\GuruFocus\guru_focus\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


[ADBE] OK


... (65 outputs left out)

[V] OK


C:\Users\mar_c\AppData\Local\Temp\ipykernel_27444\903112696.py:94: RuntimeWarning: divide by zero encountered in divide
  r = n / d
c:\Users\mar_c\OneDrive\Documentos\GuruFocus\guru_focus\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt
  result = getattr(ufunc, method)(*inputs, **kwargs)


[VRTX] OK
Ticket          Tipo  Score total último año (%)  Métodos que cumplen
   ABG no_financiera                        30.2                    2
   ACN no_financiera                        75.8                    6
  ADBE no_financiera                        70.2                    1
   ADP no_financiera                        61.0                    2
  ADSK no_financiera                        52.8                    2
   AMD no_financiera                        42.5                    1
  AMGN no_financiera                        44.2                    3
  AMZN no_financiera                        67.0                    2
   BAC         banco                        51.5                    6
   BRO no_financiera                        41.5                    7
    BX         banco                        70.9                    0
  CDNS no_financiera                        61.0                    2
    CI   aseguradora                        37.6                    3
 CMCSA no_

: 